# DSPy 101 — Replacing Prompts with Signatures

**Week 6 | Notebook 1 of 12**

**What you'll learn:**
- LM configuration (OpenAI + Ollama)
- Your first Signature — QuestionAnswering
- dspy.Predict — basic prediction
- dspy.ChainOfThought — adding rationale
- dspy.ProgramOfThought — math problems with code execution
- Side-by-side output comparison across module types
- Inline vs class-based signature syntax

**Runtime:** ~20 minutes

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("06_dspy/01_signatures_modules.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  06_dspy/01_signatures_modules.ipynb
Task:      Signatures and core modules
Calls:     ~15

With GPT-4o:       $0.15 USD
With GPT-4o-mini:  $0.01 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup & LM Configuration

In [2]:
import dspy

from src.config import get_dspy_lm, print_config

print_config()

# Configure DSPy with our unified LM
lm = get_dspy_lm()
dspy.configure(lm=lm)

print(f"\n✅ DSPy configured with: {lm.model}")

LLM Libraries — Configuration
USE_OLLAMA:        False
USE_SMALL_MODEL:   False
OLLAMA_BASE_URL:   http://localhost:11434
LLM_PROVIDER:      azureopenai
  openai model:    gpt-4o
  anthropic model: claude-opus-4-6
  gemini model:    gemini-3.7-flash
  groq model:      openai/gpt-oss-120b
  azureopenai model: gpt-4o-mini
SAMPLE_SIZE:       50
DSPY_TRIALS:       10

✅ DSPy configured with: azure/gpt-4o-mini


## 2. Your First Signature — QuestionAnswering

In [3]:
# Class-based signature (recommended for production)
class QuestionAnswering(dspy.Signature):
    """Answer questions with short factual responses."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="A concise, factual answer")


# Inline signature (quick prototyping)
# inline_sig = "question -> answer"

print("✅ Signature defined")
print(f"  Input: {QuestionAnswering.fields.keys()}")

✅ Signature defined
  Input: dict_keys(['question', 'answer'])


## 3. dspy.Predict — Basic Prediction

In [4]:
# Basic prediction module
predictor = dspy.Predict(QuestionAnswering)

result = predictor(question="What is the capital of India?")
print("Question: What is the capital of India?")
print(f"Answer: {result.answer}")

Question: What is the capital of India?
Answer: The capital of India is New Delhi.


## 4. dspy.ChainOfThought — Adding Rationale

In [5]:
# ChainOfThought automatically adds a 'reasoning' field (dspy 3.x renamed 'rationale')
cot = dspy.ChainOfThought(QuestionAnswering)

result = cot(question="If a train travels 60 km/h for 2.5 hours, how far does it go?")

print("Rationale:")
print(result.reasoning)
print(f"\nAnswer: {result.answer}")

Rationale:
To find the distance traveled by the train, we can use the formula: distance = speed × time. The train travels at a speed of 60 km/h for 2.5 hours. Therefore, the distance covered is 60 km/h × 2.5 h = 150 km.

Answer: 150 km


## 5. dspy.ProgramOfThought — Math with Code Execution